1. Importar librerías

In [ ]:
# ==========================================
# 1. Importar librerías
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)


2. Cargar dataset limpio

In [ ]:
# ==========================================
# 2. Cargar dataset limpio
# ==========================================
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
file_path = os.path.join(project_root, "data/processed/online_news_cleaned.csv")

df = pd.read_csv(file_path)
df.head()



1️⃣ Identificar y revisar las columnas LDA

In [ ]:
lda_cols = [col for col in df.columns if col.startswith("LDA_")]
df[lda_cols].head()


2️⃣ Estadísticas descriptivas básicas

In [ ]:
df[lda_cols].describe().T


3️⃣ Verificar que cada fila suma 1

In [ ]:
df["lda_sum"] = df[lda_cols].sum(axis=1)
df["lda_sum"].describe()


# 🟩 Tabla de Descripción de Variables LDA

| Variable | Descripción | Interpretación en el modelo |
|----------|-------------|------------------------------|
| **LDA_00** | Probabilidad de que el artículo pertenezca al **Tema 0** identificado por el modelo LDA. | Valores altos indican que el contenido del artículo está fuertemente asociado a las palabras clave del Tema 0. |
| **LDA_01** | Probabilidad de que el artículo pertenezca al **Tema 1** identificado por LDA. | Si este valor domina en la fila, significa que el artículo encaja mejor en el Tema 1. |
| **LDA_02** | Probabilidad asociada al **Tema 2** dentro de los temas latentes detectados. | Útil para identificar artículos con tendencia semántica hacia el Tema 2. |
| **LDA_03** | Probabilidad de pertenencia al **Tema 3** generado por LDA. | Un valor alto implica que el artículo tiene una fuerte relación temática con este tópico. |
| **LDA_04** | Probabilidad de pertenencia al **Tema 4** generado por LDA. | Representa el grado en que el artículo coincide con el patrón lingüístico del Tema 4. |


# 🟦 Tabla de Interpretación de Variables LDA Aplicada al Dataset

| Variable | Qué representa en el dataset | Cómo puede influir en Shares |
|----------|-----------------------------|-------------------------------|
| **LDA_00** | Artículos inclinados hacia el Tema 0 (contenido semántico que el modelo agrupa bajo este tópico). | Si el valor promedio de shares para este tema es alto, indica que este tipo de contenido tiende a ser más viral. |
| **LDA_01** | Artículos con alta pertenencia al Tema 1. | Puede mostrar si ciertos temas generan menos interacción; útil para detectar “temáticas frías”. |
| **LDA_02** | Contenido que el modelo clasifica dentro del Tema 2. | Si este tema tiene valores de shares muy variables, puede representar contenido altamente dependiente del contexto. |
| **LDA_03** | Artículos asociados al Tema 3. | Si el promedio de shares es bajo, podría ser un tema poco atractivo para el público. |
| **LDA_04** | Artículos con predominancia del Tema 4. | Si este tema es consistentemente viral, puede ser un predictor importante en el modelo final. |


4️⃣ Distribución de cada tema (Histogramas)

In [ ]:
df[lda_cols].hist(bins=30, figsize=(12,8))
plt.suptitle("Distribución de las Variables LDA", fontsize=16)
plt.show()


5️⃣ Boxplots para patrones y sesgos

In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(data=df[lda_cols])
plt.title("Boxplot de Variables LDA")
plt.show()


6️⃣ Heatmap de correlación entre temas

In [ ]:
plt.figure(figsize=(8,5))
sns.heatmap(df[lda_cols].corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlación entre temas LDA")
plt.show()


7️⃣ Relación entre temas (LDA) y viralidad (shares)

a) Tema dominante

In [ ]:
df["lda_topic"] = df[lda_cols].idxmax(axis=1)


b) Promedio de viralidad por tema

In [ ]:
df.groupby("lda_topic")["shares"].mean().sort_values(ascending=False)


c) Gráfica comparativa

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(
    data=df.groupby("lda_topic")["shares"].mean().reset_index(),
    x="lda_topic", y="shares"
)
plt.title("Promedio de Shares por Tema Dominante LDA")
plt.show()


8️⃣ Análisis en espacio reducido (PCA 2D)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
lda_2d = pca.fit_transform(df[lda_cols])

plt.figure(figsize=(8,6))
plt.scatter(lda_2d[:,0], lda_2d[:,1], s=5, alpha=0.5)
plt.title("Distribución de artículos en el espacio LDA (PCA 2D)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()


🟩 🔧 BLOQUE 1 — Frecuencia del Tema Dominante (muy importante)

In [ ]:
# Frecuencia de cada tema dominante
df["lda_topic"] = df[lda_cols].idxmax(axis=1)

topic_freq = df["lda_topic"].value_counts(normalize=True).sort_values(ascending=False)
topic_freq


### Frecuencia del tema dominante

El objetivo es conocer qué temas aparecen con mayor frecuencia en el dataset.  
Esto es importante para evaluar si existe sesgo temático.

- Si un tema domina → el modelo LDA asigna muchos artículos a ese tópico.
- Si un tema es raro → será menos influyente en la predicción final.

Esta inspección permite contextualizar los resultados del PCA y de la relación con los shares.


🟩 🔧 BLOQUE 2 — PCA: Varianza Explicada (indispensable)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pca.fit(df[lda_cols])

expl_var = pca.explained_variance_ratio_
expl_var


### Varianza explicada del PCA

El PCA 2D sobre las variables LDA retiene aproximadamente:

- **PC1:** XX%  
- **PC2:** XX%

En conjunto explican **(PC1 + PC2)%** de la varianza total del espacio LDA.

Esto es esperado: al proyectar un simplex de 5 dimensiones a 2D, se pierde algo de información,  
pero sigue siendo útil para visualizar patrones globales del dataset.


🟩 🔧 BLOQUE 3 — PCA 2D coloreado por tema dominante

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(lda_2d[:,0], lda_2d[:,1],
            c=df["lda_topic"].astype('category').cat.codes,
            cmap='tab10', s=5, alpha=0.6)
plt.title("PCA de Variables LDA coloreado por Tema Dominante")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()


### PCA 2D coloreado por tema dominante

El coloreado por tema dominante permite visualizar si los temas generan clústeres claros o si están mezclados.

- Si los colores se agrupan → los temas están bien diferenciados.  
- Si los colores están mezclados → los temas son difusos, lo cual es común en LDA basado en pocos tópicos (5).

En este dataset, la distribución sugiere que los temas no forman clústeres rígidos,  
sino que existe una superposición considerable entre ellos, lo cual es coherente con el comportamiento semántico del texto.


🟩 🔧 BLOQUE 4 — Pairplot opcional (útil para ver relaciones LDA entre sí)

In [ ]:
sns.pairplot(df[lda_cols], plot_kws={'alpha':0.3, 's':5})
plt.suptitle("Pairplot de Variables LDA", y=1.02)
plt.show()


### Pairplot de las variables LDA

El pairplot ayuda a observar cómo interactúan los temas entre sí.  
Dado que LDA produce proporciones que compiten entre sí, se esperan patrones como:

- Distribuciones diagonalizadas.
- Relaciones negativas entre componentes (si sube uno, bajan otros).
- Ausencia de relaciones lineales claras.

Este análisis proporciona una visualización complementaria que confirma la estructura de probabilidad simplex de las variables LDA.


# 🟩 Conclusiones del EDA LDA y Recomendaciones para el Pipeline MLOps

El análisis detallado de las variables LDA (LDA_00 – LDA_04) permite definir decisiones críticas para el diseño del pipeline de preprocesamiento dentro de la metodología MLOps. A continuación se presentan las conclusiones clave:

---

## ✅ 1. Las variables LDA no forman una distribución perfecta
Los valores muestran:
- `mean ≈ 1.003`
- `max ≈ 1.13`
- `std ≈ 0.01`

Esto indica que las probabilidades fueron redondeadas o transformadas por el dataset original.

**Impacto en MLOps:**
- No afecta modelos basados en árboles o XGBoost.
- Puede introducir ligeras inconsistencias en modelos lineales o redes neuronales.
  
**Recomendación:** incluir un paso opcional de normalización LDA dentro del pipeline para asegurar consistencia probabilística.

---

## ✅ 2. Los temas están desbalanceados
Frecuencia del tema dominante:
- LDA_04 → 24.1%
- LDA_03 → 23.2%
- LDA_02 → 21.5%
- LDA_00 → 18.6%
- LDA_01 → 12.4%

**Impacto en MLOps:**
- Es necesario documentar estas distribuciones como un baseline para detección de drift.
- No se requiere balanceo porque no es un problema de clasificación.

---

## ✅ 3. Los temas compiten entre sí (correlaciones negativas)
La matriz de correlación muestra relaciones competitivas entre temas, lo cual es natural en un espacio simplex.

**Relevancia para el pipeline:**
- No es necesario eliminar variables por multicolinealidad.
- Modelos como RandomForest, GradientBoosting y XGBoost manejan estas relaciones sin problema.

---

## ✅ 4. La relación LDA–Shares es un hallazgo clave
Ranking de viralidad por tema:
1. **LDA_03** → más viral (~4431 shares)
2. **LDA_04**
3. **LDA_01**
4. **LDA_00**
5. **LDA_02** → menos viral

**Implicaciones:**
- Las variables LDA son predictoras importantes.
- No se debe reducir su dimensionalidad en el pipeline productivo.
- Se deben monitorear cambios de comportamiento por tema como parte del tracking de performance.

---

## ✅ 5. El PCA es útil para visualización, pero no para producción
Varianza explicada:
- PC1 ≈ 32%
- PC2 ≈ 27%
- Total ≈ 59%

Esto confirma que la proyección es útil solo para exploración.

**Recomendación MLOps:**
- No aplicar PCA sobre LDA en el pipeline productivo.
- Mantener las 5 variables LDA originales por completo.

---

# 🟩 Recomendaciones para el Pipeline (fase de Feature Engineering)

### 🔹 1. Mantener las variables LDA completas (sin PCA)
Son probabilidades y contienen información semántica crucial.

### 🔹 2. Normalización opcional (recomendado)
Para corregir que no suman 1:

```python
normalize_lda = FunctionTransformer(lambda x: x / x.sum(axis=1, keepdims=True))
